## Implement the BIM (Binary Independence Model) for Nepali Documents

In [1]:
from pathlib import Path

import joblib

import nepali_pipeline  

MODEL_PATH = (Path(nepali_pipeline.__file__).resolve().parent / "nepali_hmm_pipeline.pkl")

_pipeline = joblib.load(MODEL_PATH)


def lemmatize(sentences: list[str]) -> list[list[str]]:
    """Lemmatize raw Nepali sentences using the pre-trained pipeline."""
    return _pipeline.transform(sentences)


test_data = ["केटाहरुले पोखरामा रातो स्याउ खाए।"]
print("Pipeline Output:", lemmatize(test_data))

Pipeline Output: [['केटा', 'पोखरा', 'रातो', 'स्याउ', 'खानु']]


## Step 0: Load and Lemmatize the Document Collection

Before we can rank anything, the collection of Nepali documents has to be turned into
term lists once, offline. This is the prerequisite for Algorithm 5.1 (lecture notes,
Section 5.2): every document is read from `data/`, lemmatized with the pipeline we just
tested above, and indexed so we can compute document frequencies ($n_t$) for every term.

In [2]:
from pathlib import Path

DATA_DIR = Path("data")

document_ids = []      # filenames, in a fixed order
document_texts = []    # raw text of each document, same order as document_ids

for file_path in sorted(DATA_DIR.glob("*.txt")):
    file_handle = open(file_path, encoding="utf-8", errors="ignore")
    raw_text = file_handle.read()
    file_handle.close()

    # Some files start with a UTF-8 byte-order-mark that decodes as this character.
    raw_text = raw_text.replace("\ufeff", "")

    document_ids.append(file_path.name)
    document_texts.append(raw_text)

print("Total documents loaded:", len(document_ids))

Total documents loaded: 1291


In [3]:
document_terms = []   # document_terms[i] = list of lemmatized terms for document_texts[i]

for raw_text in document_texts:
    lemmatized_output = lemmatize([raw_text])
    terms_for_this_document = lemmatized_output[0]
    document_terms.append(terms_for_this_document)

N = len(document_ids)

print("Total documents (N):", N)
print("First document, first 20 lemmatized terms:")
print(document_terms[0][:20])

Total documents (N): 1291
First document, first 20 lemmatized terms:
['प्रश्न', 'मधेशी', 'जनजातिदलहरुसंग', 'मोर्चा', 'गठन', 'गर्नु', 'हुनु', 'हुनु', 'पहिलो', 'संविधान', 'सभा', 'यस्तै', 'मोर्चा', 'ध्रुबीकरण', 'विफल', 'हुनु', 'निष्कर्ष', 'हुनु', 'कतै', 'यो']


## Step 0 (continued): Build the Document-Frequency Table ($n_t$)

For every term $t$ seen anywhere in the collection we need $n_t$ = the number of
documents that contain $t$ at least once. This is exactly the $n_t$ column of the
contingency table in Definition 5.2 / Key Formula 5.1, computed once for the whole
collection so it can be reused for every query.

In [4]:
document_frequency = {}   # term -> n_t

for terms_in_document in document_terms:
    terms_seen_in_this_document = set()
    for term in terms_in_document:
        terms_seen_in_this_document.add(term)

    for term in terms_seen_in_this_document:
        if term in document_frequency:
            document_frequency[term] = document_frequency[term] + 1
        else:
            document_frequency[term] = 1

print("Vocabulary size:", len(document_frequency))

Vocabulary size: 43713


## Phase 1, Step 1: Query Parsing and Token Extraction

The engine receives the raw Nepali query and breaks it into a set of **unique**
lemmatized search terms $Q = \{t_1, t_2, \dots, t_k\}$ (Algorithm 5.1, Step 1). We
reuse the same `lemmatize` pipeline used for the documents, so query terms and document
terms live in the same lemma space.

In [5]:
def parse_query(raw_query):
    """Algorithm 5.1, Step 1: lemmatize the query and keep only unique terms."""
    lemmatized_output = lemmatize([raw_query])
    query_terms_with_duplicates = lemmatized_output[0]

    unique_query_terms = []
    for term in query_terms_with_duplicates:
        already_in_list = False
        for existing_term in unique_query_terms:
            if existing_term == term:
                already_in_list = True
        if not already_in_list:
            unique_query_terms.append(term)

    return unique_query_terms

## Phase 1, Steps 2-3: Default Parameters and First-Pass (IDF) Weighting

With no relevance judgments yet, the system falls back on the default assumptions from
Section 5.2: $p_t = 0.5$ and $u_t \approx n_t / N$. Substituting these into Key Formula
5.1 collapses the RSJ weight to the IDF form:

$$c_t \approx \log_2\left(\frac{N - n_t}{n_t}\right)$$

A term that never occurs in the collection ($n_t = 0$) carries no discriminating
information, so its weight is set to $0$.

In [6]:
import math


def compute_initial_term_weights(query_terms, document_frequency, N):
    """Algorithm 5.1, Steps 2-3: default p_t = 0.5, u_t ~= n_t / N -> IDF-style c_t."""
    term_weights = {}

    for term in query_terms:
        if term in document_frequency:
            n_t = document_frequency[term]
        else:
            n_t = 0

        if n_t == 0:
            c_t = 0.0
        else:
            c_t = math.log2((N - n_t) / n_t)

        term_weights[term] = c_t

    return term_weights

## Phase 1, Step 4: Document Scoring and Initial Ranking

Every document's retrieval status value is the sum of the weights of the query terms it
actually contains: $RSV(D) = \sum_{t \in Q \cap D} c_t$. Documents are then ranked in
decreasing order of $RSV$.

Ranking is done here with an explicit selection procedure (repeatedly picking the
remaining document with the highest score) rather than a built-in sort, so every step
of Algorithm 5.1's ranking is visible.

In [7]:
def score_documents(document_terms, term_weights):
    """Algorithm 5.1, Step 4 / Step 8: RSV(D) = sum of c_t for matching query terms."""
    scores = []

    for terms_in_document in document_terms:
        terms_present = set()
        for term in terms_in_document:
            terms_present.add(term)

        document_score = 0.0
        for term in term_weights:
            if term in terms_present:
                document_score = document_score + term_weights[term]

        scores.append(document_score)

    return scores


def rank_documents(scores, top_k):
    """Return the top_k (document_index, score) pairs, highest score first.

    Implemented as an explicit repeated-selection loop instead of a built-in sort.
    """
    remaining_indices = []
    for index in range(len(scores)):
        remaining_indices.append(index)

    number_to_pick = top_k
    if number_to_pick > len(remaining_indices):
        number_to_pick = len(remaining_indices)

    ranked_results = []
    for _ in range(number_to_pick):
        best_position_in_remaining = 0
        best_index = remaining_indices[0]
        best_score = scores[best_index]

        for position in range(len(remaining_indices)):
            candidate_index = remaining_indices[position]
            if scores[candidate_index] > best_score:
                best_score = scores[candidate_index]
                best_index = candidate_index
                best_position_in_remaining = position

        ranked_results.append((best_index, best_score))
        remaining_indices.pop(best_position_in_remaining)

    return ranked_results

## Running Phase 1 (Initial Pass) on a Real Query

We now run Steps 1-4 end to end on a Nepali query over the actual document collection.

In [8]:
raw_query = "संविधान सभा निर्वाचन"

query_terms = parse_query(raw_query)
print("Lemmatized, unique query terms:", query_terms)

initial_term_weights = compute_initial_term_weights(query_terms, document_frequency, N)
print("\nPhase 1 (IDF) term weights:")
for term in initial_term_weights:
    print(" ", term, "-> c_t =", round(initial_term_weights[term], 4))

initial_scores = score_documents(document_terms, initial_term_weights)
top_results_phase1 = rank_documents(initial_scores, top_k=10)

print("\nTop results after Phase 1 (Initial Pass):")
for rank_position in range(len(top_results_phase1)):
    document_index, score = top_results_phase1[rank_position]
    print(rank_position + 1, "-", document_ids[document_index], "  RSV =", round(score, 4))

Lemmatized, unique query terms: ['संविधान', 'सभा', 'निर्वाचन']

Phase 1 (IDF) term weights:
  संविधान -> c_t = 3.8449
  सभा -> c_t = 5.9183
  निर्वाचन -> c_t = 4.7584

Top results after Phase 1 (Initial Pass):
1 - 1419532741.txt   RSV = 14.5216
2 - 1422952461.txt   RSV = 14.5216
3 - 1423103285.txt   RSV = 14.5216
4 - 1423104695.txt   RSV = 14.5216
5 - 1426765059.txt   RSV = 14.5216
6 - 1445960123.txt   RSV = 14.5216
7 - 1451448360.txt   RSV = 10.6767
8 - 1452410369.txt   RSV = 10.6767
9 - 1454910720.txt   RSV = 10.6767
10 - 1405693439.txt   RSV = 9.7632


## Phase 2, Step 5: Capturing Relevance Feedback (Pseudo-Relevance Feedback)

Once Phase 1 has produced a ranked list, the system needs real relevance counts $R$ and
$r_t$ to move beyond the IDF guess. Here we use **pseudo-relevance feedback (PRF)**: the
top-$K$ documents from Phase 1 are simply assumed to be relevant, without asking the
user for explicit judgments.

In [9]:
def gather_pseudo_relevance_feedback(top_results):
    """Algorithm 5.1, Step 5 (PRF branch): assume the Phase 1 top results are relevant."""
    relevant_document_indices = []
    for document_index, score in top_results:
        relevant_document_indices.append(document_index)

    R = len(relevant_document_indices)
    return relevant_document_indices, R


def count_relevant_term_frequency(query_terms, relevant_document_indices, document_terms):
    """Algorithm 5.1, Step 5: r_t = number of assumed-relevant documents containing t."""
    relevant_term_frequency = {}
    for term in query_terms:
        relevant_term_frequency[term] = 0

    for document_index in relevant_document_indices:
        terms_present = set()
        for term in document_terms[document_index]:
            terms_present.add(term)

        for term in query_terms:
            if term in terms_present:
                relevant_term_frequency[term] = relevant_term_frequency[term] + 1

    return relevant_term_frequency

## Phase 2, Steps 6-7: Re-estimating $p_t$, $u_t$ and Recomputing the Full RSJ Weight

With $R$ and $r_t$ now known, the default assumptions are replaced by empirical
estimates, smoothed by $+0.5$ to avoid division by zero when $r_t = 0$ or $r_t = R$
(Section 5.2):

$$p_t = \frac{r_t + 0.5}{R + 1.0}, \qquad u_t = \frac{n_t - r_t + 0.5}{N - R + 1.0}$$

and the full Key Formula 5.1 is applied:

$$c_t = \log_2\left(\frac{p_t (1 - u_t)}{u_t (1 - p_t)}\right)$$

In [10]:
def compute_updated_term_weights(query_terms, document_frequency, relevant_term_frequency, R, N):
    """Algorithm 5.1, Steps 6-7: smoothed p_t, u_t, then the full RSJ log-odds weight."""
    updated_term_weights = {}

    for term in query_terms:
        if term in document_frequency:
            n_t = document_frequency[term]
        else:
            n_t = 0

        r_t = relevant_term_frequency[term]

        p_t = (r_t + 0.5) / (R + 1.0)
        u_t = (n_t - r_t + 0.5) / (N - R + 1.0)

        c_t = math.log2((p_t * (1 - u_t)) / (u_t * (1 - p_t)))

        updated_term_weights[term] = c_t

    return updated_term_weights

## Phase 2, Step 8: Re-scoring and Re-ranking

Finally, every document in the collection is re-scored with the updated $c_t$ values
and re-ranked, reusing the exact same `score_documents` and `rank_documents` functions
from Phase 1. Documents that scored poorly under plain IDF can now rise if they contain
terms whose weight was boosted by the feedback.

In [11]:
relevant_document_indices, R = gather_pseudo_relevance_feedback(top_results_phase1)
relevant_term_frequency = count_relevant_term_frequency(query_terms, relevant_document_indices, document_terms)

print("Pseudo-relevance feedback: R =", R)
print("r_t counts:", relevant_term_frequency)

updated_term_weights = compute_updated_term_weights(
    query_terms, document_frequency, relevant_term_frequency, R, N
)

print("\nPhase 2 (full RSJ) term weights:")
for term in updated_term_weights:
    print(" ", term, "-> c_t =", round(updated_term_weights[term], 4))

updated_scores = score_documents(document_terms, updated_term_weights)
top_results_phase2 = rank_documents(updated_scores, top_k=10)

print("\nTop results after Phase 2 (Feedback Update Cycle):")
for rank_position in range(len(top_results_phase2)):
    document_index, score = top_results_phase2[rank_position]
    print(rank_position + 1, "-", document_ids[document_index], "  RSV =", round(score, 4))

Pseudo-relevance feedback: R = 10
r_t counts: {'संविधान': 7, 'सभा': 10, 'निर्वाचन': 9}

Phase 2 (full RSJ) term weights:
  संविधान -> c_t = 5.0576
  सभा -> c_t = 11.1799
  निर्वाचन -> c_t = 7.7155

Top results after Phase 2 (Feedback Update Cycle):
1 - 1419532741.txt   RSV = 23.9531
2 - 1422952461.txt   RSV = 23.9531
3 - 1423103285.txt   RSV = 23.9531
4 - 1423104695.txt   RSV = 23.9531
5 - 1426765059.txt   RSV = 23.9531
6 - 1445960123.txt   RSV = 23.9531
7 - 1451448360.txt   RSV = 18.8954
8 - 1452410369.txt   RSV = 18.8954
9 - 1454910720.txt   RSV = 18.8954
10 - 1405693439.txt   RSV = 16.2376


## Inspecting a Top-Ranked Document

As a sanity check, print the first few hundred characters of the top-ranked document
from Phase 2, to confirm it is actually about the query topic.

In [12]:
top_document_index, top_score = top_results_phase2[0]
print("Top document:", document_ids[top_document_index])
print("RSV score:", round(top_score, 4))
print()
print(document_texts[top_document_index][:500])

Top document: 1419532741.txt
RSV score: 23.9531


तपाईको राजनैतिक यात्रा र पृष्ठभुमि बारे केही बताईदिनुस न ?
मेरो राजनैतिक यात्रा आज भन्दा ठिक सात वर्ष अघि बाट शुरुवात भएको हो । त्यो भन्दा पहिला राजनैतिक तर्फ रुचि राख्ने गर्दथे तर यस्तो राजनैतिक प्लेटफर्मको खोजीमा या पर्खाईमा थिए, जहाँ म आफुलाई समाहित गर्न सक्दथे । हुन त नेपालमा राजनैतिक दलहरुको कमि त थिएनन तै पनि म फरक विचार धारा भएको हुदाँ कुनै न कुनै नयाँ उदेश्यमा रुची राख्ने भएकोले तिन प्रमुख कुराले मलाई प्रेरित बनायो, त्यो हो ।  सहि समय सहि स्थान  म यसबाट अभिप्रेरित भई नेपालको पहिलो संविध
